# Distributional RL (C51, QR-DQN) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def discount(rewards,gamma=0.9):
    return sum((gamma**t)*r for t,r in enumerate(rewards))
def moving_avg(x,n=5):
    x=np.asarray(x,dtype=float); return np.convolve(x,np.ones(n)/n,mode='valid')
print('setup ok')

## Learning a return distribution instead of only its mean
Distributional RL keeps the spread of possible returns, not just their average.

In [ ]:
# 1) return distribution keeps atoms, not just a mean
atoms=np.array([-1.,0.,1.,2.]); p=np.array([0.1,0.2,0.5,0.2]); mean=(atoms*p).sum()
plt.figure(figsize=(6,3)); plt.bar(atoms,p,width=.25); plt.title(f'distributional value mean = {mean:.2f}')
assert abs(mean-0.8)<1e-12 and abs(p.sum()-1)<1e-12

In [ ]:
# 2) quantiles describe uneven outcome spread
q=np.array([0.1,0.5,0.9]); vals=np.array([-0.5,0.8,2.0])
plt.figure(figsize=(6,3)); plt.plot(q,vals,'-o'); plt.xlabel('quantile'); plt.ylabel('return'); plt.title('QR-DQN-style return quantiles')
assert vals[2]>vals[1]>vals[0]

In [ ]:
# 3) C51 projection clips shifted atoms to fixed support
support=np.linspace(-1,2,4); shifted=np.clip(0.5+0.9*support,support.min(),support.max())
plt.figure(figsize=(6,3)); plt.plot(support,shifted,'o-'); plt.xlabel('old atom'); plt.ylabel('target atom'); plt.title('reward plus discounted support')
assert shifted[-1]==2.0

In [ ]:
# 4) two policies can have same mean and different risk
atoms=np.array([0.,1.,2.]); p_safe=np.array([0.1,0.8,0.1]); p_risky=np.array([0.5,0.0,0.5])
plt.figure(figsize=(6,3)); plt.plot(atoms,p_safe,'-o',label='safe'); plt.plot(atoms,p_risky,'-s',label='risky'); plt.legend(); plt.title('same mean, different distribution')
assert abs((atoms*p_safe).sum()-(atoms*p_risky).sum())<1e-12

In [ ]:
# 5) categorical cross-entropy trains the whole distribution
p=np.array([0.1,0.2,0.5,0.2]); target=np.array([0.0,0.1,0.6,0.3]); ce=-(target*np.log(p)).sum()
plt.figure(figsize=(6,3)); plt.bar(range(4),target-p); plt.title(f'target minus prediction, CE={ce:.3f}')
assert abs(ce-1.0596634733096073)<1e-12